# Traffic Data Preprocessing Pipeline

This notebook performs comprehensive data cleaning and filtering on traffic detection data.

## Pipeline Stages:

1. **Data Loading**: Load parquet files from date range
2. **Lifetime Filtering**: Remove short-lived detections (noise/false positives)
3. **Footpath Zone Filtering**: Remove vehicles in pedestrian-only zones
4. **Crosswalk Zone Filtering**: Remove vehicles moving parallel to crosswalks
5. **Static Object Removal**: Remove stationary vehicles (parked cars, etc.)

## Output:
Clean dataset ready for near-miss analysis (DRAC, TTC, etc.)

- pedestrian=1
- bicycle=2
- motorcycle=3
- car=4
- escooter=5,
- van=6
- truck=7
- bus=8

In [1]:
%reset -f

In [2]:
# imports
import pandas as pd
import os
from datetime import datetime, timedelta
import numpy as np
import time
from tqdm import tqdm
import gc
import psutil

In [3]:
# Memory Monitoring Utilities
def log_memory(label=""):
    """Quick memory snapshot of current process"""
    process = psutil.Process()
    mem_mb = process.memory_info().rss / 1024 / 1024
    print(f"[MEMORY] {label}: {mem_mb:.1f} MB")
    return mem_mb

def log_df_memory(df, name="DataFrame"):
    """Log DataFrame memory usage"""
    mem_mb = df.memory_usage(deep=True).sum() / 1024 / 1024
    print(f"[DF MEMORY] {name}: {mem_mb:.1f} MB ({len(df):,} rows)")
    return mem_mb

In [4]:
START_DATE = "2025-06-01"
END_DATE = "2025-06-01"

DATA_DIR = "/home/ubuntu/data/uploads/objects/clean"
CHUNK_SIZE = 500000

In [5]:
def load_data(data_dir, start_date, end_date):
    """
    Load data with optimized dtypes for memory efficiency
    """
    # Define optimal dtypes upfront - saves ~50% memory
    dtypes = {
        'id': 'int32',           # Was int64 → Save 50%
        'label': 'int8',         # Values 1-8 → Save 87.5%
        'pos_x': 'float32',      # Was float64 → Save 50%
        'pos_y': 'float32',
        'pos_z': 'float32',
        'vel': 'float32',
        'vel_x': 'float32',
        'vel_y': 'float32',
        'yaw': 'float32',
        'size_x': 'float32',
        'size_y': 'float32',
        # timestamp stays int64 (needed for precision)
    }
    
    # List to collect dataframes
    dfs = []
    
    # Loop over all folders within date range
    for folder in tqdm(sorted(os.listdir(data_dir)), desc="Loading data"):
        folder_path = os.path.join(data_dir, folder)
        
        # Skip non-folders
        if not os.path.isdir(folder_path):
            continue
        
        if folder.startswith(start_date) or folder.startswith(end_date):
            # Load all parquet files inside the folder
            df_chunk = pd.read_parquet(folder_path)
            
            # Apply dtypes to matching columns
            for col, dtype in dtypes.items():
                if col in df_chunk.columns:
                    df_chunk[col] = df_chunk[col].astype(dtype)
            
            dfs.append(df_chunk)
            del df_chunk  # Clean up immediately
    
    # Combine all into single dataframe
    if dfs:
        log_memory("Before concat")
        df = pd.concat(dfs, ignore_index=True)
        
        # Destroy the list immediately
        del dfs
        gc.collect()
        
        log_memory("After concat + cleanup")
        log_df_memory(df, "Loaded data")
        return df
    else:
        print("No data found for given date range.")
        return pd.DataFrame()

# Usage
df = load_data(DATA_DIR, START_DATE, END_DATE)
print(f"Loaded {len(df)} records from {START_DATE} to {END_DATE}")

Loading data: 100%|██████████| 336/336 [00:03<00:00, 97.81it/s]


[MEMORY] Before concat: 4132.9 MB
[MEMORY] After concat + cleanup: 3874.7 MB
[DF MEMORY] Loaded data: 2273.1 MB (39,074,272 rows)
Loaded 39074272 records from 2025-06-01 to 2025-06-01


In [6]:
df.head()

,timestamp,id,label,pos_x,pos_y,pos_z,size_x,size_y,size_z,vel_x,vel_y,vel_z,yaw,yaw_rate,vel
0,2025-06-01 00:00:00.037209034,8748438,4,61.968750,-58.531250,-0.109985,4.269531,2.009766,1.769531,0.010002,-0.010002,0.0,5.386719,0.0,0.0
1,2025-06-01 00:00:00.037209034,9896407,4,6.921875,25.187500,0.099976,4.589844,2.150391,1.769531,0.000000,-0.000000,0.0,5.417969,0.0,0.0
2,2025-06-01 00:00:00.037209034,10035255,4,-20.265625,56.687500,-0.049988,4.191406,1.990234,1.839844,-0.010002,0.010002,0.0,2.285156,-0.0,0.0
3,2025-06-01 00:00:00.037209034,10049521,6,10.726562,20.984375,-0.020004,5.570312,2.240234,1.940430,-0.000000,0.000000,0.0,2.349609,0.0,0.0
4,2025-06-01 00:00:00.037209034,10096097,4,-25.500000,29.359375,0.239990,4.359375,2.099609,1.879883,0.020004,-0.020004,0.0,5.511719,-0.0,0.0


In [7]:
obj_id = 8748438
df_id = df[df["id"] == obj_id].sort_values("timestamp").reset_index(drop=True)
df_id.head()

,timestamp,id,label,pos_x,pos_y,pos_z,size_x,size_y,size_z,vel_x,vel_y,vel_z,yaw,yaw_rate,vel
0,2025-06-01 00:00:00.037209034,8748438,4,61.96875,-58.53125,-0.109985,4.269531,2.009766,1.769531,0.010002,-0.010002,0.0,5.386719,0.0,0.000000
1,2025-06-01 00:00:00.136997938,8748438,4,61.96875,-58.53125,-0.109985,4.269531,2.009766,1.769531,0.000000,-0.000000,0.0,5.386719,0.0,0.000000
2,2025-06-01 00:00:00.229187965,8748438,4,61.96875,-58.53125,-0.109985,4.269531,2.009766,1.769531,0.010002,-0.010002,0.0,5.378906,-0.0,0.000000
3,2025-06-01 00:00:00.329210997,8748438,4,61.96875,-58.56250,-0.109985,4.269531,2.009766,1.769531,0.010002,-0.020004,0.0,5.375000,-0.0,0.078107
4,2025-06-01 00:00:00.443810940,8748438,4,61.96875,-58.53125,-0.109985,4.269531,2.009766,1.769531,0.010002,-0.020004,0.0,5.371094,-0.0,0.007948


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39074272 entries, 0 to 39074271
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int8          
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(1), int8(1)
memory usage: 2.2 GB


In [9]:
len(df['timestamp'])

39074272

In [10]:
df['timestamp'].duplicated().sum()

np.int64(38210512)

In [11]:
df.reset_index(drop=True, inplace=True)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39074272 entries, 0 to 39074271
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int8          
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(1), int8(1)
memory usage: 2.2 GB


### 1. Lifetime Filtering

In [13]:
# Global minimum detection count required per label
global_lifespan_thresholds = {
        1: 30,
        2: 80,
        3: 60,
        4: 90,
        5: 30,
        6: 100,
        7: 100,   
        8: 180
    }

log_memory("Before lifetime filtering")

# Compute lifespan as detection count in full dataset
lifespan = (
    df.groupby(["id", "label"])["timestamp"]
    .count()
    .reset_index(name="lifespan")
)

# Attach thresholds
lifespan["min_required"] = lifespan["label"].map(global_lifespan_thresholds)

# Identify short-lived objects
lifespan["is_outlier"] = lifespan["lifespan"] < lifespan["min_required"]

# Get outlier IDs
short_lived_ids = set(lifespan.loc[lifespan["is_outlier"], "id"].tolist())

# Destroy lifespan DataFrame immediately
del lifespan
gc.collect()

# Drop them from the cleaned dataframe
df = df[~df["id"].isin(short_lived_ids)]

# Logging
print(f"[lifespan filter] Removed {len(short_lived_ids)} short-lived IDs")

# Clean up IDs set
del short_lived_ids
gc.collect()

log_df_memory(df, "After lifetime filter")

[MEMORY] Before lifetime filtering: 3877.5 MB


[lifespan filter] Removed 3157 short-lived IDs
[DF MEMORY] After lifetime filter: 2564.0 MB (38,964,803 rows)


np.float64(2564.0214986801147)

In [14]:
df.reset_index(inplace = True, drop = True)

### 2. Footpath Zone Filtering

In [15]:
footpath_zones = [{"id":"1081","name":"FalseDetection (Vehicles as Pedestrians)","type":"analytics","vertices":"POLYGON ((-8.6426068 4.1825497, -5.2591289 6.6106291, 2.4873278 -1.8810115, 3.0701297 -4.1885040, -4.8739637 -7.3121410, -5.9190415 -5.1056689, -14.9469623 -8.3614314, -15.4947729 -6.7531838, -6.4707399 -3.4197283, -5.5463181 -0.1021501, -8.6426068 4.1825497))","min_z":-1.5,"max_z":3.5},
{"id":"1082","name":"FalseDetection (Vehicle as Pedestrian)","type":"analytics","vertices":"POLYGON ((-3.3894790 -13.3168241, 11.5404031 -7.8269771, 18.9154074 -17.2223685, 14.9261910 -19.9865761, 21.4404136 -27.6760383, 20.0765345 -28.6872835, 7.1069287 -14.4841637, -11.9112838 -20.7401988, -12.4875105 -18.6473010, -2.5169133 -15.3193791, -3.3894790 -13.3168241))","min_z":-1.5,"max_z":3.5},
{"id":"1083","name":"FalseDetection (Vehicles as Pedestrians)","type":"analytics","vertices":"POLYGON ((65.0702212 14.5604360, 36.5624449 3.1898636, 35.8594607 -0.3697733, 48.5353798 -17.2319024, 52.8105665 -14.6263596, 49.1754262 -9.8991876, 52.2197357 2.2111523, 68.4919414 9.0673664, 65.0702212 14.5604360))","min_z":-1.5,"max_z":3.5},
{"id":"1084","name":"FalseDetection (Vehicles as Pedestrians)","type":"analytics","vertices":"POLYGON ((5.9894856 29.2443364, 7.5000886 30.3643575, 15.4508444 22.7984588, 22.8276900 37.1368332, 27.9001775 34.9101554, 20.8424398 19.4127592, 17.0919999 16.0917894, 5.9894856 29.2443364))","min_z":-1.5,"max_z":3.5}]

import geopandas as gpd
from shapely import wkt

zones_df = pd.DataFrame(footpath_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")

In [16]:
def attach_zones_to_objects(df, gdf_zones, how="inner", batch_size=100000):
    """
    Attach zone information to objects via spatial join.
    Handles duplicates when objects span multiple zones.
    """
    columns = df.columns.tolist()
    num_chunks = len(df) // batch_size + 1
    output_chunks = []

    for i in range(num_chunks):
        chunk = df.iloc[i*batch_size : (i+1)*batch_size].copy()

        if len(chunk) == 0:
            continue

        gdf_chunk = gpd.GeoDataFrame(
            chunk,
            geometry=gpd.points_from_xy(chunk["pos_x"], chunk["pos_y"]),
        )

        joined = gpd.sjoin(gdf_chunk, gdf_zones, how=how, predicate="within")
        
        # 🔥 CRITICAL: Drop geometry immediately after join!
        if 'geometry' in joined.columns:
            joined = joined.drop(columns=['geometry'])
        
        # Destroy gdf_chunk right away
        del gdf_chunk

        # Handle empty spatial joins
        if len(joined) == 0:
            if how == "left":
                chunk["zone"] = np.nan
                output_chunks.append(chunk)
            del joined  # Clean up
            continue

        # Rename columns
        joined = joined.rename(columns={
            "id_left": "id",
            "id_right": "zone"
        })

        # Remove duplicates (objects in multiple zones - keep first)
        joined = joined.drop_duplicates(subset=['id', 'timestamp'], keep='first')
        
        # Select only needed columns
        joined = joined[columns + ["zone"]]
        
        # Convert zone to category dtype (saves memory)
        joined['zone'] = joined['zone'].astype('category')

        output_chunks.append(joined)
        
        # Clean up this iteration
        del joined
        
        # Force garbage collection every 5 batches
        if i % 5 == 0:
            gc.collect()

    # Handle case where no objects are in zones
    if len(output_chunks) == 0:
        result = df.copy()
        result["zone"] = np.nan
        return result

    result = pd.concat(output_chunks, ignore_index=True)
    
    # Destroy output_chunks
    del output_chunks
    gc.collect()
    
    return result

In [17]:
# Attach footpath zones in chunks
total_rows = len(df)
processed_chunks = []

print(f"Attaching footpath zones to {total_rows} rows...")
log_memory("Before footpath zones")

for i in tqdm(range(0, total_rows, CHUNK_SIZE), desc="Attaching footpath zones"):
    chunk = df.iloc[i:i+CHUNK_SIZE].copy()
    processed_chunks.append(attach_zones_to_objects(chunk, gdf_zones, how="left"))
    del chunk  # Clean up chunk

df = pd.concat(processed_chunks, ignore_index=True)

# Destroy the list immediately
del processed_chunks
gc.collect()

log_memory("After footpath zones")
print(f"✓ Zones attached! Total rows: {len(df)}")

Attaching footpath zones to 38964803 rows...
[MEMORY] Before footpath zones: 5969.9 MB


Attaching footpath zones: 100%|██████████| 78/78 [00:34<00:00,  2.25it/s]


[MEMORY] After footpath zones: 8128.7 MB
✓ Zones attached! Total rows: 38964803


In [18]:
def apply_footpath_zone_filter(df):
    """
    Remove vehicles that shouldn't be in footpath zones based on:
    1. Speed limits per vehicle type
    2. Forbidden vehicle types (trucks, buses)
    Memory-optimized with aggressive cleanup.
    """
    max_speed = {
        1: 4.0,   # pedestrian
        2: 6.0,   # bicycle
        3: 12.0,  # motorcycle
        4: 12.0,  # car
        5: 4.0,   # escooter
        6: 12.0,  # van
        7: 0.0,   # truck - forbidden
        8: 0.0,   # bus - forbidden
    }

    df_zone = df[df["zone"].notnull()].copy()
    speed_limit_series = df_zone["label"].map(max_speed)

    # Vehicles that shouldn't be in footpath zones
    forbidden_mask = df_zone["label"].isin([3, 4, 5, 6, 7, 8])

    # Vehicles exceeding speed limits
    speed_exceed_mask = df_zone["vel"] > speed_limit_series

    remove_mask = forbidden_mask | speed_exceed_mask

    removed_ids = df_zone.loc[remove_mask, "id"].unique()
    
    # Clean up intermediate variables immediately
    del df_zone, speed_limit_series, forbidden_mask, speed_exceed_mask, remove_mask
    
    df = df.loc[~df["id"].isin(removed_ids)].copy()

    print(f"[footpath zone] Removed {len(removed_ids)} objects")
    
    # Clean up IDs
    del removed_ids
    gc.collect()

    return df

In [19]:
# Apply footpath zone filter
df = apply_footpath_zone_filter(df)

[footpath zone] Removed 703 objects


In [20]:
# Clean up: drop zone column and reset index
df = df.drop(columns=["zone"])
df = df.reset_index(drop=True)

# Force cleanup after dropping geometry-related column
gc.collect()
log_memory("After footpath filter cleanup")

[MEMORY] After footpath filter cleanup: 7317.5 MB


7317.51171875

In [21]:
# Verify data after footpath filtering
print(f"Remaining objects: {len(df)}")
print(f"Unique IDs: {df['id'].nunique()}")

Remaining objects: 38289116
Unique IDs: 52737


### 3. Crosswalk Zone Filtering

In [22]:
crosswalk_zones = [
    {"id":"1015","name":"Crosswalk Houba - South","type":"analytics","vertices":"POLYGON ((25.0 -23.6, 42.8 -8.5, 40.3 -5.5, 22.6 -21.0, 25.0 -23.6))","min_z":-1.5,"max_z":3.5},
    {"id":"1016","name":"Crosswalk Amandiers","type":"analytics","vertices":"POLYGON ((-2.7 -13.4, -4.8 -7.4, -1.4 -6.0, 0.7 -12.2, -2.7 -13.4))","min_z":-1.5,"max_z":3.5},
    {"id":"1017","name":"Crosswalk Houba - North","type":"analytics","vertices":"POLYGON ((-3.1 4.7, 13.8 19.3, 16.9 16.1, 0.0 1.4, -3.1 4.7))","min_z":-1.5,"max_z":3.5},
    {"id":"1018","name":"Crosswalk Magnolias","type":"analytics","vertices":"POLYGON ((30.5 15.5, 32.2 18.9, 23.4 23.5, 21.6 20.2, 30.5 15.5))","min_z":-1.5,"max_z":3.5},
    {"id":"1019","name":"Crosswalk Charlotte [1]","type":"analytics","vertices":"POLYGON ((36.4 15.3, 40.4 5.2, 37.1 3.8, 33.1 14.0, 36.4 15.3))","min_z":-1.5,"max_z":3.5}
]

zones_df = pd.DataFrame(crosswalk_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")

In [23]:
def compute_polygon_orientation(poly):
    """
    Calculate the orientation of a polygon based on its longest edge.
    Returns angle in degrees.
    """
    coords = list(poly.exterior.coords)
    edges = []

    for (x1, y1), (x2, y2) in zip(coords, coords[1:]):
        dx, dy = x2 - x1, y2 - y1
        length = (dx**2 + dy**2) ** 0.5
        edges.append((length, dx, dy))

    # Get longest edge
    _, dx, dy = max(edges, key=lambda e: e[0])
    return np.degrees(np.arctan2(dy, dx))

In [25]:
def filter_parallel_vehicles(df_zone, orientation_deg, threshold=4.0):
    """
    Filter vehicles moving parallel to crosswalk axis.
    Vehicles traveling along the crosswalk (not crossing) are removed.
    
    Args:
        df_zone: DataFrame with objects in a single zone
        orientation_deg: Crosswalk orientation in degrees
        threshold: Maximum angle difference (degrees) to consider parallel
    
    Returns:
        parallel_ids: List of IDs to remove
        df_zone_filtered: Filtered DataFrame
    """
    # Filter all vehicle types (exclude pedestrians)
    vehicle_labels = [3, 4, 6, 7, 8]  # from motorcycle through bus except for the escooter
    vehicles = df_zone[df_zone["label"].isin(vehicle_labels)].copy()

    if vehicles.empty:
        return [], df_zone

    # Compute vehicle heading from yaw
    vehicles["heading_deg"] = np.degrees(vehicles["yaw"])

    # Fallback: use velocity direction if yaw is missing
    missing = vehicles["heading_deg"].isna()
    vehicles.loc[missing, "heading_deg"] = np.degrees(
        np.arctan2(vehicles.loc[missing, "vel_y"], vehicles.loc[missing, "vel_x"])
    )

    # Calculate angular difference
    def angle_diff(a, b):
        d = (a - b + 180) % 360 - 180
        return abs(d)

    vehicles["angle_diff"] = vehicles.apply(
        lambda r: angle_diff(r["heading_deg"], orientation_deg),
        axis=1
    )

    # Find vehicles moving parallel to crosswalk
    parallel = vehicles[vehicles["angle_diff"] < threshold]["id"].unique().tolist()

    # Remove parallel vehicles
    df_zone_filtered = df_zone[~df_zone["id"].isin(parallel)]

    return parallel, df_zone_filtered

In [26]:
# Precompute zone orientations
gdf_zones["orientation_deg"] = gdf_zones["geometry"].apply(compute_polygon_orientation)

removed_ids_global = []
processed_chunks = []

# Attach crosswalk zones in chunks
total_rows = len(df)
print(f"Attaching crosswalk zones to {total_rows} rows...")
log_memory("Before crosswalk zones")

for i in tqdm(range(0, total_rows, CHUNK_SIZE), desc="Attaching crosswalk zones"):
    chunk = df.iloc[i:i+CHUNK_SIZE].copy()
    processed_chunk = attach_zones_to_objects(chunk, gdf_zones, how="left")
    processed_chunks.append(processed_chunk)
    del chunk  # Clean up chunk

df = pd.concat(processed_chunks, ignore_index=True)

# Destroy chunks immediately
del processed_chunks
gc.collect()

log_memory("After crosswalk zone attachment")
print(f"✓ Zones attached! Total rows: {len(df)}")

# Check zone distribution
print(f"Objects in zones: {df['zone'].notna().sum()}")
print(f"Objects outside zones: {df['zone'].isna().sum()}")

Attaching crosswalk zones to 38289116 rows...
[MEMORY] Before crosswalk zones: 7317.6 MB


Attaching crosswalk zones: 100%|██████████| 77/77 [00:33<00:00,  2.31it/s]


[MEMORY] After crosswalk zone attachment: 8538.9 MB
✓ Zones attached! Total rows: 38289116
Objects in zones: 792138
Objects outside zones: 37496978


In [27]:
# Process each crosswalk zone separately
for zone_id, zone_df in tqdm(df.groupby("zone"), desc="Filtering parallel vehicles"):
    # Skip NaN zones (objects outside all crosswalk zones)
    if pd.isna(zone_id):
        print(f"Skipping {len(zone_df)} objects outside all crosswalk zones")
        continue
    
    print(f"Processing crosswalk zone {zone_id} with {len(zone_df)} objects")

    zone_orientation = float(
        gdf_zones.loc[gdf_zones["id"] == str(zone_id), "orientation_deg"].iloc[0]
    )

    removed_ids, zone_filtered = filter_parallel_vehicles(
        zone_df, orientation_deg=zone_orientation, threshold=4.0
    )

    removed_ids_global.append(removed_ids)
    print(f"  → Removed {len(removed_ids)} parallel vehicles")
    
    # Clean up iteration variables
    del removed_ids, zone_filtered

# Flatten list and remove IDs
removed_ids_global = [item for sublist in removed_ids_global for item in sublist]
df = df[~df["id"].isin(removed_ids_global)]

print(f"\n✓ Total parallel vehicles removed: {len(removed_ids_global)}")
print(f"✓ Remaining objects: {len(df)}")

# Clean up IDs list
del removed_ids_global
gc.collect()

log_memory("After crosswalk filtering")

Filtering parallel vehicles:   0%|          | 0/5 [00:00<?, ?it/s]

Processing crosswalk zone 1015 with 265019 objects


Filtering parallel vehicles:  20%|██        | 1/5 [00:02<00:09,  2.40s/it]

  → Removed 2 parallel vehicles
Processing crosswalk zone 1016 with 71078 objects
  → Removed 1 parallel vehicles
Processing crosswalk zone 1017 with 254382 objects


Filtering parallel vehicles:  60%|██████    | 3/5 [00:02<00:01,  1.21it/s]

  → Removed 1 parallel vehicles
Processing crosswalk zone 1018 with 99796 objects
  → Removed 0 parallel vehicles
Processing crosswalk zone 1019 with 101863 objects


Filtering parallel vehicles: 100%|██████████| 5/5 [00:03<00:00,  1.58it/s]

  → Removed 0 parallel vehicles



✓ Total parallel vehicles removed: 4
✓ Remaining objects: 38287948
[MEMORY] After crosswalk filtering: 10699.1 MB


10699.12890625

In [28]:
# Clean up: drop zone column and reset index
df = df.drop(columns=["zone"])
df = df.reset_index(drop=True)

# Force cleanup after dropping zone column
gc.collect()
log_memory("After crosswalk filter cleanup")

[MEMORY] After crosswalk filter cleanup: 10407.0 MB


10407.01171875

In [29]:
# Verify data after crosswalk filtering
print(f"Remaining objects: {len(df)}")
print(f"Unique IDs: {df['id'].nunique()}")

Remaining objects: 38287948
Unique IDs: 52733


### 4. Remove Static Objects

In [30]:
STATIC_THRESHOLD = 0.5     # m/s - velocity below this is considered static
STATIC_RATIO_MIN = 0.8     # 80% of lifetime must be static

log_memory("Before static filter")

# Build per-object velocity history
df_vel = (
    df.groupby(["id", "label"])["vel"]
    .apply(list)
    .reset_index()
)

# Compute lifespan
df_vel["lifespan"] = df_vel["vel"].apply(len)

# Count static frames
df_vel["static_frames"] = df_vel["vel"].apply(
    lambda v: sum(vi < STATIC_THRESHOLD for vi in v)
)

# Calculate static ratio
df_vel["static_ratio"] = df_vel["static_frames"] / df_vel["lifespan"]

# Flag static objects
df_vel["is_static"] = df_vel["static_ratio"] >= STATIC_RATIO_MIN

# Get IDs to remove
removable_static_ids = set(
    df_vel[df_vel["is_static"]]["id"].astype(int).tolist()
)

print(f"[static filter] Found {len(removable_static_ids)} static objects")

# Destroy df_vel immediately
del df_vel
gc.collect()

# Remove static objects
df = df[~df['id'].isin(removable_static_ids)]

print(f"[static filter] Removed {len(removable_static_ids)} static objects")
print(f"Remaining objects: {len(df)}")

# Clean up IDs set
del removable_static_ids
gc.collect()

log_memory("After static filter")
log_df_memory(df, "After static filter")

[MEMORY] Before static filter: 10407.0 MB


[static filter] Found 6672 static objects
[static filter] Removed 6672 static objects
Remaining objects: 14994565
[MEMORY] After static filter: 9343.8 MB
[DF MEMORY] After static filter: 986.7 MB (14,994,565 rows)


np.float64(986.6952753067017)

In [31]:
# Final cleanup
df = df.reset_index(drop=True)
gc.collect()
log_memory("After final cleanup")

[MEMORY] After final cleanup: 9343.8 MB


9343.8046875

In [32]:
# Final verification
print("="*60)
print("FINAL DATASET SUMMARY")
print("="*60)
print(f"Total objects: {len(df)}")
print(f"Unique IDs: {df['id'].nunique()}")
print(f"Unique timestamps: {df['timestamp'].nunique()}")
print(f"\nLabel distribution:")
print(df['label'].value_counts().sort_index())
print("="*60)

FINAL DATASET SUMMARY
Total objects: 14994565
Unique IDs: 46061
Unique timestamps: 846344

Label distribution:
label
1    4716668
2     172748
3     146746
4    8928679
5      63197
6     674627
7      92015
8     199885
Name: count, dtype: int64


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14994565 entries, 0 to 14994564
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int8          
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(1), int8(1)
memory usage: 872.3 MB


In [34]:
df.reset_index(inplace = True, drop = True)

In [36]:
# =============================================================================
# DATA QUALITY FILTERS (Teleportation & Jitter) - Run on df
# =============================================================================

print(f"Initial rows: {len(df):,}")

df = df.sort_values(['id', 'timestamp'])

# TELEPORTATION FILTER
df['_dt'] = df.groupby('id')['timestamp'].diff().dt.total_seconds()
df['_dv'] = df.groupby('id')['vel'].diff()
df['_acc'] = (df['_dv'] / df['_dt']).fillna(0).abs()
teleport_mask = df['_acc'] > 10.0
print(f"-> Teleportation glitches: {teleport_mask.sum():,}")

# JITTER FILTER
df['_prev_yaw'] = df.groupby('id')['yaw'].shift(1)
yaw_diff = (df['yaw'] - df['_prev_yaw']).fillna(0).abs()
yaw_diff = np.where(yaw_diff > np.pi, 2*np.pi - yaw_diff, yaw_diff)
jitter_mask = (yaw_diff > 0.785) & (df['vel'] > 2.0)
print(f"-> Jitter glitches: {jitter_mask.sum():,}")

# APPLY
bad_rows = teleport_mask | jitter_mask
df = df[~bad_rows].copy()
df.drop(columns=['_dt', '_dv', '_acc', '_prev_yaw'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Removed {bad_rows.sum():,} rows. Remaining: {len(df):,}")
gc.collect()

Initial rows: 14,994,565
-> Teleportation glitches: 49,460
-> Jitter glitches: 18,507
Removed 67,062 rows. Remaining: 14,927,503


769

In [37]:
len(df)

14927503

# Data extraction - Near miss pairs (Heavy - Heavy)

*Current zone data, pedastrian crossing zones, is for Light to Heavy vehicles only!

---

## Method-Specific Pair Filtering

**Architecture:**
1. **Generic pairing** (`utils.py`): Distance + speed filters → generates all nearby pairs
2. **Lane-based filtering** (here): Apply method-specific criteria
   - **MDRAC**: Same-lane longitudinal only (car-following scenarios)
   - **SPF**: All conflict types (lane changes, intersections, perpendicular approaches)

**Rationale:** MDRAC assumes longitudinal car-following. Lane geometry prevents false positives from:
- Vehicles turning at intersections (different lanes)
- Parallel vehicles on adjacent lanes (not following)
- Cross-traffic at intersections (perpendicular motion)

---

In [38]:
# Define the Zones
near_miss_zones = [{"id":"1074","name":"Pedestrian Crossing 1","type":"analytics","vertices":"POLYGON ((24.2738305 11.0336169, 31.7356793 13.5560292, 36.2078141 15.1576709, 43.7450150 17.4969908, 47.2622737 8.1635769, 36.9982138 3.5232094, 32.5602804 1.6666404, 27.3267870 -0.3062503, 24.2738305 11.0336169))","min_z":-1.5,"max_z":3.5},
{"id":"1075","name":"Pedestrian Crossing 2","type":"analytics","vertices":"POLYGON ((28.4322135 35.4097216, 37.6101806 31.0730297, 29.7613022 13.8943178, 24.0457174 12.0937765, 19.1230952 14.6240664, 28.4322135 35.4097216))","min_z":-1.5,"max_z":3.5},
{"id":"1076","name":"Pedestrian Crossing 3","type":"analytics","vertices":"POLYGON ((-0.4008521 23.7028977, 3.8500930 26.7534852, 11.9475235 17.4708113, 13.9290183 18.9399734, 22.7019680 11.3504234, 15.4719481 5.8276760, -0.4008521 23.7028977))","min_z":-1.5,"max_z":3.5},
{"id":"1077","name":"Pedestrian Crossing 4","type":"analytics","vertices":"POLYGON ((-13.4857933 16.0361501, -2.3947097 3.8495495, 2.2686744 -1.0889511, 11.2978547 7.4421036, 7.8841631 11.7104214, 5.8591757 11.8061846, -5.7526480 23.8885602, -13.4857933 16.0361501))","min_z":-1.5,"max_z":3.5},
{"id":"1078","name":"Pedestrian Crossing 5","type":"analytics","vertices":"POLYGON ((3.3656059 -4.4085345, -24.1838846 -14.1766537, -22.1521598 -19.7115374, 6.1376057 -10.0653907, 3.3656059 -4.4085345))","min_z":-1.5,"max_z":3.5},
{"id":"1079","name":"Pedestrian Crossing 6","type":"analytics","vertices":"POLYGON ((18.1027860 -15.9690219, 24.7117391 -12.1747441, 30.5032592 -18.9617192, 28.5317011 -21.0059641, 40.8463196 -36.3820050, 37.9033636 -39.0948896, 18.1027860 -15.9690219))","min_z":-1.5,"max_z":3.5},
{"id":"1080","name":"Pedestrian Crossing 7","type":"analytics","vertices":"POLYGON ((25.6180338 -11.6782049, 42.4498255 -32.1848405, 52.3394447 -23.7967882, 36.5273842 -0.7475620, 25.6180338 -11.6782049))","min_z":-1.5,"max_z":3.5},
]

zones_df = pd.DataFrame(near_miss_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")

# Filter objects inside zones (using existing attach_zones_to_objects function)
objects = attach_zones_to_objects(df, gdf_zones, how="inner")
print(f"✓ {len(objects):,} rows in pedestrian crossing zones")

✓ 4,927,455 rows in pedestrian crossing zones


### Lane Geometry & Method-Specific Filtering

In [ ]:
# Load lane geometry data (similar structure to crossing zones)

lane_geometries = [
]

lanes_df = pd.DataFrame(lane_geometries)
lanes_df["geometry"] = lanes_df["vertices"].apply(wkt.loads)
gdf_lanes = gpd.GeoDataFrame(lanes_df, geometry="geometry")

print(f"✓ Loaded {len(gdf_lanes)} lane geometries")
print(f"  Lane types: {gdf_lanes['direction'].unique()}")

In [ ]:
def attach_lane_to_pairs(pairs_df, lanes_gdf, id_col1='id1', id_col2='id2'):
    """
    Attach lane information to vehicle pairs using spatial join.
    
    Args:
        pairs_df: DataFrame with vehicle pairs (from utils.find_vehicle_vehicle_pairs)
        lanes_gdf: GeoDataFrame with lane geometries
        id_col1: Column name for first vehicle ID
        id_col2: Column name for second vehicle ID
        
    Returns:
        DataFrame with lane_id1, lane_id2, and same_lane columns added
    """
    # Create GeoDataFrames for each vehicle position
    gdf1 = gpd.GeoDataFrame(
        pairs_df,
        geometry=gpd.points_from_xy(pairs_df['pos1_x'], pairs_df['pos1_y'])
    )
    gdf2 = gpd.GeoDataFrame(
        pairs_df,
        geometry=gpd.points_from_xy(pairs_df['pos2_x'], pairs_df['pos2_y'])
    )
    
    # Spatial join to find which lane each vehicle is in
    joined1 = gpd.sjoin(gdf1, lanes_gdf[['id', 'geometry']], how='left', predicate='within')
    joined2 = gpd.sjoin(gdf2, lanes_gdf[['id', 'geometry']], how='left', predicate='within')
    
    # Add lane info back to pairs (drop geometry to return regular DataFrame)
    result = pairs_df.copy()
    result['lane_id1'] = joined1['id_right'].values
    result['lane_id2'] = joined2['id_right'].values
    result['same_lane'] = result['lane_id1'] == result['lane_id2']
    
    return result

In [ ]:
# All existing zones
{"id":"1001","name":"Detection Range","type":"detection","vertices":"POLYGON ((-40.0 -60.0, 90.0 -60.0, 90.0 70.0, -40.0 70.0, -40.0 -60.0))","min_z":-1.5,"max_z":3.5}
{"id":"1002","name":"calibration 0","type":"calibration","vertices":"POLYGON ((0.5 0.4, 0.4 -0.6, -0.4 -0.6, -0.3 0.4, 0.5 0.4))","min_z":-1.5,"max_z":3.5}
{"id":"1003","name":"calibration 1","type":"calibration","vertices":"POLYGON ((29.0 52.1, 16.3 28.5, 11.4 28.5, 8.7 31.5, 29.0 52.1))","min_z":-1.5,"max_z":1.0}
{"id":"1004","name":"calibration 2","type":"calibration","vertices":"POLYGON ((51.6 -10.5, 54.1 0.2, 56.7 1.3, 51.6 -10.5))","min_z":-1.5,"max_z":1.0}
{"id":"1005","name":"calibration 3","type":"calibration","vertices":"POLYGON ((-22.0 -8.9, -6.5 -3.2, -5.6 0.1, -13.9 9.6, -22.0 -8.9))","min_z":-1.5,"max_z":1.0}
{"id":"1006","name":"calibration 4","type":"calibration","vertices":"POLYGON ((44.7 -58.5, 12.2 -20.3, 8.2 -20.2, -1.4 -24.1, 44.7 -58.5))","min_z":-1.5,"max_z":1.0}
{"id":"1007","name":"exclusion 0","type":"exclusion","vertices":"POLYGON ((-40.0 -60.0, 47.4 -60.4, 12.4 -20.1, 7.2 -15.4, 4.8 -14.9, -40.2 -31.5, -40.0 -60.0))","min_z":-1.5,"max_z":3.5}
{"id":"1008","name":"exclusion 1","type":"exclusion","vertices":"POLYGON ((61.5 -29.8, 86.8 -59.8, 89.9 -59.9, 90.0 15.0, 53.9 0.6, 51.2 -10.6, 65.3 -26.6, 61.5 -29.8))","min_z":-1.5,"max_z":3.5}
{"id":"1009","name":"exclusion 2","type":"exclusion","vertices":"POLYGON ((90.0 70.0, 90.0 44.9, 61.5 33.6, 50.0 45.5, 63.5 71.0, 90.0 70.0))","min_z":-1.5,"max_z":3.5}
{"id":"1010","name":"exclusion 3","type":"exclusion","vertices":"POLYGON ((-40.0 -15.3, -20.1 -8.3, -17.9 -5.0, -8.1 -1.7, -6.7 1.7, -14.4 10.4, -13.7 11.1, -18.1 16.1, -40.4 41.1, -40.0 -15.3))","min_z":-1.5,"max_z":3.5}
{"id":"1011","name":"exclusion 4","type":"exclusion","vertices":"POLYGON ((39.2 70.5, 46.1 70.6, 36.2 53.5, 33.3 59.9, 16.5 28.4, 11.3 28.3, 8.7 31.3, 4.1 34.4, -27.6 71.0, 39.2 70.5))","min_z":-1.5,"max_z":3.5}
{"id":"1012","name":"calibration 5","type":"calibration","vertices":"POLYGON ((44.3 -8.7, 43.7 -8.7, 43.7 -7.8, 44.2 -7.8, 44.3 -8.7))","min_z":-1.5,"max_z":3.5}
{"id":"1013","name":"calibration 6","type":"calibration","vertices":"POLYGON ((19.5 17.4, 19.4 17.1, 19.1 17.3, 19.2 17.6, 19.5 17.4))","min_z":-1.5,"max_z":3.5}
{"id":"1014","name":"calibration 7","type":"calibration","vertices":"POLYGON ((25.7 -16.4, 25.8 -16.7, 25.6 -16.8, 25.5 -16.5, 25.7 -16.4))","min_z":-1.5,"max_z":3.5}
{"id":"1015","name":"Crosswalk Houba - South","type":"analytics","vertices":"POLYGON ((25.0 -23.6, 42.8 -8.5, 40.3 -5.5, 22.6 -21.0, 25.0 -23.6))","min_z":-1.5,"max_z":3.5}
{"id":"1016","name":"Crosswalk Amandiers","type":"analytics","vertices":"POLYGON ((-2.7 -13.4, -4.8 -7.4, -1.4 -6.0, 0.7 -12.2, -2.7 -13.4))","min_z":-1.5,"max_z":3.5}
{"id":"1017","name":"Crosswalk Houba - North","type":"analytics","vertices":"POLYGON ((-3.1 4.7, 13.8 19.3, 16.9 16.1, 0.0 1.4, -3.1 4.7))","min_z":-1.5,"max_z":3.5}
{"id":"1018","name":"Crosswalk Magnolias","type":"analytics","vertices":"POLYGON ((30.5 15.5, 32.2 18.9, 23.4 23.5, 21.6 20.2, 30.5 15.5))","min_z":-1.5,"max_z":3.5}
{"id":"1019","name":"Crosswalk Charlotte [1]","type":"analytics","vertices":"POLYGON ((36.4 15.3, 40.4 5.2, 37.1 3.8, 33.1 14.0, 36.4 15.3))","min_z":-1.5,"max_z":3.5}
{"id":"1020","name":"Road Houba South Ext-Int","type":"analytics","vertices":"POLYGON ((51.4 -21.5, 53.1 -24.5, 57.7 -30.1, 51.6 -35.2, 34.7 -15.7, 42.8 -8.8, 51.4 -21.5))","min_z":-1.5,"max_z":3.5}
{"id":"1021","name":"Road Houba South Int-Ext [1]","type":"analytics","vertices":"POLYGON ((48.5 -37.7, 51.5 -35.3, 34.6 -15.8, 31.6 -18.3, 48.5 -37.7))","min_z":-1.5,"max_z":3.5}
{"id":"1022","name":"Road Houba South Int-Ext [2]","type":"analytics","vertices":"POLYGON ((42.1 -43.0, 44.8 -40.8, 28.2 -21.3, 25.4 -23.7, 42.1 -43.0))","min_z":-1.5,"max_z":3.5}
{"id":"1023","name":"Road Amandiers Ext-Int","type":"analytics","vertices":"POLYGON ((-22.9 -17.6, -22.2 -19.7, -3.2 -12.9, -4.1 -10.4, -22.9 -17.6))","min_z":-1.5,"max_z":3.5}
{"id":"1024","name":"Road Houba North Ext-Int","type":"analytics","vertices":"POLYGON ((-9.5 28.4, -16.2 22.2, -1.7 6.1, 5.3 12.2, -9.5 28.4))","min_z":-1.5,"max_z":3.5}
{"id":"1025","name":"Road Houba North Ext-Int","type":"analytics","vertices":"POLYGON ((10.7 16.8, -3.7 33.3, -6.6 30.6, 7.9 14.3, 10.7 16.8))","min_z":-1.5,"max_z":3.5}
{"id":"1026","name":"Road Magnolias Ext-Int [1]","type":"analytics","vertices":"POLYGON ((30.3 34.6, 34.4 32.6, 28.4 21.2, 24.5 23.2, 30.3 34.6))","min_z":-1.5,"max_z":3.5}
{"id":"1027","name":"Road Charlotte-Magnolias","type":"analytics","vertices":"POLYGON ((49.5 28.4, 45.3 24.7, 40.4 30.1, 44.4 34.0, 49.5 28.4))","min_z":-1.5,"max_z":3.5}
{"id":"1028","name":"Road Magnolias Int-Ext","type":"analytics","vertices":"POLYGON ((38.0 30.7, 32.3 19.1, 28.6 21.0, 34.6 32.5, 38.0 30.7))","min_z":-1.5,"max_z":3.5}
{"id":"1029","name":"Road Charlotte Ext-Int [1]","type":"analytics","vertices":"POLYGON ((51.5 14.2, 38.8 9.6, 36.6 15.0, 49.3 19.5, 51.5 14.2))","min_z":-1.5,"max_z":3.5}
{"id":"1030","name":"Road Charlotte Int-Ext [1]","type":"analytics","vertices":"POLYGON ((53.3 9.9, 40.5 5.4, 38.9 9.4, 51.6 14.0, 53.3 9.9))","min_z":-1.5,"max_z":3.5}
{"id":"1031","name":"Road Amandiers Int-Ext","type":"analytics","vertices":"POLYGON ((-24.0 -14.5, -23.0 -17.5, -4.1 -10.3, -5.1 -7.5, -24.0 -14.5))","min_z":-1.5,"max_z":3.5}
{"id":"1032","name":"Intersection","type":"analytics","vertices":"POLYGON ((21.9 19.6, 30.2 15.2, 32.7 13.8, 36.8 3.6, 35.6 -0.2, 40.0 -5.2, 22.5 -20.6, 11.4 -7.8, 5.9 -9.6, 2.8 -2.0, 0.4 1.2, 17.3 16.1, 21.9 19.6))","min_z":-1.5,"max_z":3.5}
{"id":"1033","name":"Road Magnolias Ext-Int [2]","type":"analytics","vertices":"POLYGON ((39.0 52.0, 43.2 49.8, 38.5 40.9, 34.5 43.0, 39.0 52.0))","min_z":-1.5,"max_z":3.5}
{"id":"1034","name":"Road Magnolias Int-Ext [2]","type":"analytics","vertices":"POLYGON ((43.3 49.7, 47.3 47.5, 42.8 38.6, 38.6 40.8, 43.3 49.7))","min_z":-1.5,"max_z":3.5}
{"id":"1035","name":"Road Charlotte Ext-Int [2]","type":"analytics","vertices":"POLYGON ((63.3 24.7, 65.3 19.6, 57.7 16.7, 55.8 22.0, 63.3 24.7))","min_z":-1.5,"max_z":3.5}
{"id":"1036","name":"Road Charlotte Int-Ext [2]","type":"analytics","vertices":"POLYGON ((67.0 15.2, 59.5 12.2, 57.8 16.6, 65.3 19.5, 67.0 15.2))","min_z":-1.5,"max_z":3.5}
{"id":"1037","name":"Bike Houba South [2]","type":"analytics","vertices":"POLYGON ((39.4 -11.8, 41.0 -10.5, 57.6 -30.0, 56.1 -31.3, 39.4 -11.8))","min_z":-1.5,"max_z":3.5}
{"id":"1038","name":"Bike Houba South [1]","type":"analytics","vertices":"POLYGON ((30.5 -19.3, 31.6 -18.4, 48.5 -37.8, 47.3 -38.7, 30.5 -19.3))","min_z":-1.5,"max_z":3.5}
{"id":"1039","name":"Bike Houba North [1]","type":"analytics","vertices":"POLYGON ((-18.0 20.6, -16.4 22.1, -1.8 6.0, -3.3 4.7, -18.0 20.6))","min_z":-1.5,"max_z":3.5}
{"id":"1040","name":"Road Houba Int-Int","type":"analytics","vertices":"POLYGON ((-12.2 36.3, -7.2 31.0, -9.8 28.7, -14.8 34.0, -12.2 36.3))","min_z":-1.5,"max_z":3.5}
{"id":"1041","name":"Bike Houba North [2]","type":"analytics","vertices":"POLYGON ((-2.5 34.3, 11.8 17.7, 10.9 16.9, -3.6 33.3, -2.5 34.3))","min_z":-1.5,"max_z":3.5}
{"id":"1042","name":"Road Houba North Ext-Int [2]","type":"analytics","vertices":"POLYGON ((-28.2 35.0, -24.2 38.8, -17.4 31.8, -21.5 27.9, -28.2 35.0))","min_z":-1.5,"max_z":3.5}
{"id":"1043","name":"Road Houba North Int-Ext [2]","type":"analytics","vertices":"POLYGON ((-18.6 44.2, -15.4 46.9, -8.9 39.6, -12.0 36.8, -18.6 44.2))","min_z":-1.5,"max_z":3.5}
{"id":"1044","name":"static 0","type":"static","vertices":"POLYGON ((82.7 -59.7, 86.6 -59.9, 62.2 -30.8, 60.0 -32.7, 82.7 -59.7))","min_z":-1.5,"max_z":3.5}
{"id":"1045","name":"static 1","type":"static","vertices":"POLYGON ((61.0 -59.8, 64.3 -59.7, 35.9 -26.3, 34.2 -27.9, 61.0 -59.8))","min_z":-1.5,"max_z":3.5}
{"id":"1046","name":"static 2","type":"static","vertices":"POLYGON ((53.5 -59.9, 56.6 -59.9, 27.9 -26.8, 26.2 -28.2, 53.5 -59.9))","min_z":-1.5,"max_z":3.5}
{"id":"1047","name":"static 3","type":"static","vertices":"POLYGON ((-3.5 -13.2, -39.9 -26.1, -40.1 -28.3, -2.7 -14.9, -3.5 -13.2))","min_z":-1.5,"max_z":3.5}
{"id":"1048","name":"static 4","type":"static","vertices":"POLYGON ((-39.9 -20.4, -39.9 -17.7, -6.0 -5.3, -5.2 -7.4, -39.9 -20.4))","min_z":-1.5,"max_z":3.5}
{"id":"1049","name":"static 5","type":"static","vertices":"POLYGON ((-39.9 43.1, -39.8 46.8, -21.0 27.0, -23.2 25.1, -39.9 43.1))","min_z":-1.5,"max_z":3.5}
{"id":"1050","name":"static 6","type":"static","vertices":"POLYGON ((-32.9 69.9, -29.9 69.9, 13.8 19.4, 11.9 17.7, -32.9 69.9))","min_z":-1.5,"max_z":3.5}
{"id":"1051","name":"exclusion 5","type":"exclusion","vertices":"POLYGON ((-40.0 67.8, -14.6 39.1, -14.0 36.6, -16.4 33.9, -18.4 33.0, -40.0 56.1, -40.0 67.8))","min_z":-1.5,"max_z":3.5}
{"id":"1052","name":"Crosswalk Charlotte [2]","type":"analytics","vertices":"POLYGON ((46.7 23.1, 51.2 27.3, 54.1 24.1, 49.3 20.1, 46.7 23.1))","min_z":-1.5,"max_z":3.5}
{"id":"1053","name":"exclusion 6","type":"exclusion","vertices":"POLYGON ((46.4 -9.1, 49.6 -12.9, 48.0 -14.4, 44.8 -10.5, 46.4 -9.1))","min_z":-1.5,"max_z":3.5}
{"id":"1054","name":"exclusion 7","type":"exclusion","vertices":"POLYGON ((20.0 -29.4, 23.8 -25.9, 27.5 -30.3, 23.6 -33.6, 20.0 -29.4))","min_z":-1.5,"max_z":3.5}
{"id":"1055","name":"exclusion 8","type":"exclusion","vertices":"POLYGON ((-9.7 28.8, -7.2 31.0, 7.8 14.3, 5.4 12.3, -9.7 28.8))","min_z":-1.5,"max_z":3.5}
{"id":"1056","name":"exclusion 9","type":"exclusion","vertices":"POLYGON ((41.9 5.3, 61.0 12.5, 62.2 8.9, 47.7 2.8, 48.9 -0.2, 44.5 -1.8, 41.9 5.3))","min_z":-1.5,"max_z":3.5}
{"id":"1057","name":"exclusion 10","type":"exclusion","vertices":"POLYGON ((32.0 50.3, 34.8 50.7, 22.9 27.7, 20.3 28.1, 32.0 50.3))","min_z":-1.5,"max_z":3.5}
{"id":"1058","name":"Exclusion Parking Limit","type":"exclusion","vertices":"POLYGON ((-40.0 -26.2, -32.4 -23.5, -31.8 -25.4, -40.1 -28.5, -40.0 -26.2))","min_z":-1.5,"max_z":3.5}
{"id":"1059","name":"Exclusion Parking Limit","type":"exclusion","vertices":"POLYGON ((-39.8 46.8, -33.2 39.9, -35.0 37.7, -39.9 43.0, -39.8 46.8))","min_z":-1.5,"max_z":3.5}
{"id":"1060","name":"Default Name","type":"exclusion","vertices":"POLYGON ((46.8 -38.8, 44.6 -40.7, 41.2 -36.8, 43.3 -34.8, 46.8 -38.8))","min_z":2.0,"max_z":3.5}
{"id":"1061","name":"Default Name","type":"exclusion","vertices":"POLYGON ((45.2 -38.0, 44.0 -39.2, 43.1 -38.2, 44.2 -37.0, 45.2 -38.0))","min_z":-1.5,"max_z":3.5}
{"id":"1062","name":"Default Name","type":"exclusion","vertices":"POLYGON ((56.2 -50.7, 55.0 -51.8, 53.5 -50.1, 54.6 -49.0, 56.2 -50.7))","min_z":-1.5,"max_z":3.5}
{"id":"1063","name":"Default Name","type":"exclusion","vertices":"POLYGON ((45.6 33.9, 45.4 33.7, 51.7 26.7, 52.0 26.9, 45.6 33.9))","min_z":-1.5,"max_z":3.5}
{"id":"1064","name":"Default Name","type":"exclusion","vertices":"POLYGON ((36.4 -39.4, 35.0 -40.6, 33.8 -39.3, 35.2 -38.1, 36.4 -39.4))","min_z":-1.5,"max_z":3.5}
{"id":"1065","name":"Default Name","type":"exclusion","vertices":"POLYGON ((28.1 -30.3, 27.5 -30.8, 26.9 -30.1, 27.5 -29.6, 28.1 -30.3))","min_z":-1.5,"max_z":3.5}
{"id":"1066","name":"Default Name","type":"exclusion","vertices":"POLYGON ((80.7 42.7, 85.5 32.8, 63.5 24.1, 59.7 34.3, 80.7 42.7))","min_z":2.0,"max_z":3.5}
{"id":"1067","name":"Default Name","type":"exclusion","vertices":"POLYGON ((87.5 -97.3, 83.9 -100.7, 15.1 -20.4, 18.7 -17.3, 87.5 -97.3))","min_z":2.0,"max_z":3.5}
{"id":"1068","name":"Default Name","type":"exclusion","vertices":"POLYGON ((41.4 28.6, 46.7 22.6, 48.9 19.7, 40.1 16.2, 36.6 15.3, 32.0 14.6, 31.8 15.1, 32.6 19.2, 33.9 22.1, 38.7 30.4, 41.4 28.6))","min_z":2.099609375,"max_z":3.5}
{"id":"1069","name":"Default Name","type":"exclusion","vertices":"POLYGON ((66.9 26.1, 58.2 22.9, 58.1 23.3, 66.8 26.5, 66.9 26.1))","min_z":-1.5,"max_z":3.5}
{"id":"1070","name":"Default Name","type":"exclusion","vertices":"POLYGON ((66.3 4.4, 65.0 7.9, 53.5 3.2, 54.8 -0.2, 66.3 4.4))","min_z":2.5,"max_z":3.5}
{"id":"1071","name":"Parking Space 0","type":"analytics","vertices":"POLYGON ((-11.3812755 -7.3877609, -10.5282730 -9.4876865, -5.2482486 -7.4328254, -6.1443975 -5.3139289, -11.3812755 -7.3877609))","min_z":-1.5,"max_z":3.5}
{"id":"1072","name":"Parking Space 1","type":"analytics","vertices":"POLYGON ((-10.6040479 -9.4835855, -15.2094212 -11.2161382, -15.9231174 -8.9229975, -11.3613318 -7.3928900, -10.6040479 -9.4835855))","min_z":-1.5,"max_z":3.5}
{"id":"1073","name":"Parking Space 2","type":"analytics","vertices":"POLYGON ((-15.3316291 -11.1921562, -16.0782430 -8.9702262, -21.3812789 -10.9300679, -20.5290255 -13.1467122, -15.3316291 -11.1921562))","min_z":-1.5,"max_z":3.5}
{"id":"1074","name":"Pedestrian Crossing 1","type":"analytics","vertices":"POLYGON ((24.2738305 11.0336169, 31.7356793 13.5560292, 36.2078141 15.1576709, 43.7450150 17.4969908, 47.2622737 8.1635769, 36.9982138 3.5232094, 32.5602804 1.6666404, 27.3267870 -0.3062503, 24.2738305 11.0336169))","min_z":-1.5,"max_z":3.5}
{"id":"1075","name":"Pedestrian Crossing 2","type":"analytics","vertices":"POLYGON ((28.4322135 35.4097216, 37.6101806 31.0730297, 29.7613022 13.8943178, 24.0457174 12.0937765, 19.1230952 14.6240664, 28.4322135 35.4097216))","min_z":-1.5,"max_z":3.5}
{"id":"1076","name":"Pedestrian Crossing 3","type":"analytics","vertices":"POLYGON ((-0.4008521 23.7028977, 3.8500930 26.7534852, 11.9475235 17.4708113, 13.9290183 18.9399734, 22.7019680 11.3504234, 15.4719481 5.8276760, -0.4008521 23.7028977))","min_z":-1.5,"max_z":3.5}
{"id":"1077","name":"Pedestrian Crossing 4","type":"analytics","vertices":"POLYGON ((-13.4857933 16.0361501, -2.3947097 3.8495495, 2.2686744 -1.0889511, 11.2978547 7.4421036, 7.8841631 11.7104214, 5.8591757 11.8061846, -5.7526480 23.8885602, -13.4857933 16.0361501))","min_z":-1.5,"max_z":3.5}
{"id":"1078","name":"Pedestrian Crossing 5","type":"analytics","vertices":"POLYGON ((3.3656059 -4.4085345, -24.1838846 -14.1766537, -22.1521598 -19.7115374, 6.1376057 -10.0653907, 3.3656059 -4.4085345))","min_z":-1.5,"max_z":3.5}
{"id":"1079","name":"Pedestrian Crossing 6","type":"analytics","vertices":"POLYGON ((18.1027860 -15.9690219, 24.7117391 -12.1747441, 30.5032592 -18.9617192, 28.5317011 -21.0059641, 40.8463196 -36.3820050, 37.9033636 -39.0948896, 18.1027860 -15.9690219))","min_z":-1.5,"max_z":3.5}
{"id":"1080","name":"Pedestrian Crossing 7","type":"analytics","vertices":"POLYGON ((25.6180338 -11.6782049, 42.4498255 -32.1848405, 52.3394447 -23.7967882, 36.5273842 -0.7475620, 25.6180338 -11.6782049))","min_z":-1.5,"max_z":3.5}
{"id":"1081","name":"FalseDetection (Vehicles as Pedestrians)","type":"analytics","vertices":"POLYGON ((-8.6426068 4.1825497, -5.2591289 6.6106291, 2.4873278 -1.8810115, 3.0701297 -4.1885040, -4.8739637 -7.3121410, -5.9190415 -5.1056689, -14.9469623 -8.3614314, -15.4947729 -6.7531838, -6.4707399 -3.4197283, -5.5463181 -0.1021501, -8.6426068 4.1825497))","min_z":-1.5,"max_z":3.5}
{"id":"1082","name":"FalseDetection (Vehicle as Pedestrian)","type":"analytics","vertices":"POLYGON ((-3.3894790 -13.3168241, 11.5404031 -7.8269771, 18.9154074 -17.2223685, 14.9261910 -19.9865761, 21.4404136 -27.6760383, 20.0765345 -28.6872835, 7.1069287 -14.4841637, -11.9112838 -20.7401988, -12.4875105 -18.6473010, -2.5169133 -15.3193791, -3.3894790 -13.3168241))","min_z":-1.5,"max_z":3.5}
{"id":"1083","name":"FalseDetection (Vehicles as Pedestrians)","type":"analytics","vertices":"POLYGON ((65.0702212 14.5604360, 36.5624449 3.1898636, 35.8594607 -0.3697733, 48.5353798 -17.2319024, 52.8105665 -14.6263596, 49.1754262 -9.8991876, 52.2197357 2.2111523, 68.4919414 9.0673664, 65.0702212 14.5604360))","min_z":-1.5,"max_z":3.5}
{"id":"1084","name":"FalseDetection (Vehicles as Pedestrians)","type":"analytics","vertices":"POLYGON ((5.9894856 29.2443364, 7.5000886 30.3643575, 15.4508444 22.7984588, 22.8276900 37.1368332, 27.9001775 34.9101554, 20.8424398 19.4127592, 17.0919999 16.0917894, 5.9894856 29.2443364))","min_z":-1.5,"max_z":3.5}
 

In [ ]:
# =============================================================================
# EXTRACT VEHICLE-VEHICLE PAIRS (Generic - from utils.py)
# =============================================================================
import sys
sys.path.append('/home/ubuntu/prem')

from ssm.utils import load_config, find_vehicle_vehicle_pairs
from tqdm import tqdm
import io, contextlib

config = load_config('/home/ubuntu/prem/config.yaml')

# Process in chunks to generate base pairs
unique_times = objects['timestamp'].unique()
chunk_size = 1000
all_pairs = []

print("Step 1: Extracting all nearby pairs (generic)")
for i in tqdm(range(0, len(unique_times), chunk_size), desc="Extracting pairs"):
    time_chunk = unique_times[i:i+chunk_size]
    df_chunk = objects[objects['timestamp'].isin(time_chunk)]
    
    with contextlib.redirect_stdout(io.StringIO()):
        pairs_chunk = find_vehicle_vehicle_pairs(df_chunk, config)
    
    if len(pairs_chunk) > 0:
        all_pairs.append(pairs_chunk)
    
    del df_chunk
    gc.collect()

vehicle_pairs = pd.concat(all_pairs, ignore_index=True) if all_pairs else pd.DataFrame()
print(f"✓ Extracted {len(vehicle_pairs):,} vehicle-vehicle pairs")

Extracting pairs:   0%|          | 0/798 [00:00<?, ?it/s]

Extracting pairs: 100%|██████████| 798/798 [01:53<00:00,  7.03it/s]

✓ Extracted 3,972 vehicle-vehicle pairs


In [ ]:
# =============================================================================
# APPLY METHOD-SPECIFIC FILTERING (Lane-based)
# =============================================================================
print("\nStep 2: Applying lane-based filtering for MDRAC")

if len(vehicle_pairs) > 0:
    # Attach lane information to pairs using spatial join
    vehicle_pairs_with_lanes = attach_lane_to_pairs(vehicle_pairs, gdf_lanes)
    
    # Filter for MDRAC (same-lane longitudinal only)
    mdrac_pairs = filter_pairs_for_mdrac(vehicle_pairs_with_lanes)
    
    print(f"✓ Prepared {len(mdrac_pairs):,} pairs for MDRAC analysis")
else:
    mdrac_pairs = pd.DataFrame()
    print("No pairs to filter")

In [ ]:
# =============================================================================
# CALCULATE MDRAC NEAR-MISS SCORES
# =============================================================================
# Applies Modified DRAC formula with Perception-Reaction Time

from ssm.m_drac import ModifiedDRAC

print("\nStep 3: Calculating MDRAC scores")

if len(mdrac_pairs) > 0:
    # Initialize MDRAC detector
    mdrac_detector = ModifiedDRAC(config)
    
    # Calculate MDRAC for same-lane pairs only
    mdrac_pairs = mdrac_detector.calculate_mdrac(mdrac_pairs)
    
    # Filter by minimum threshold
    min_mdrac = config['mdrac']['min_mdrac']
    conflicts = mdrac_pairs[mdrac_pairs['mdrac'] >= min_mdrac].copy()
    
    # Classify severity
    conflicts = mdrac_detector.classify_severity(conflicts)
    
    # Deduplicate (keep first detection of each pair)
    conflicts = mdrac_detector.deduplicate_temporal(conflicts)
    
    # Format output
    conflicts = mdrac_detector.format_output(conflicts)
    
    print(f"✓ {len(conflicts):,} near-miss conflicts detected (MDRAC >= {min_mdrac})")
    if len(conflicts) > 0:
        print(f"  Severity: {conflicts['severity'].value_counts().to_dict()}")
else:
    conflicts = pd.DataFrame()
    print("⚠ No pairs to analyze")

✓ 604 near-miss conflicts detected (MDRAC >= 3.4)
  Severity: {'severe': 350, 'critical': 183, 'normal': 45, 'moderate': 26}


In [42]:
# =============================================================================
# SAVE RESULTS TO CSV
# =============================================================================

output_path = '/home/ubuntu/prem/results/mdrac_conflicts.csv'

if len(conflicts) > 0:
    conflicts.to_csv(output_path, index=False)
    print(f"✓ Saved {len(conflicts):,} conflicts to {output_path}")
    
    # Preview
    print(f"\nTop 5 most critical (lowest TTC):")
    display(conflicts.nsmallest(5, 'ttc')[['timestamp', 'interaction', 'distance', 'ttc', 'mdrac', 'severity']])
else:
    print("⚠ No conflicts to save")

✓ Saved 604 conflicts to /home/ubuntu/prem/results/mdrac_conflicts.csv

Top 5 most critical (lowest TTC):


,timestamp,interaction,distance,ttc,mdrac,severity
401,2025-06-01 16:42:03.520311117,car_car,3.965613,0.263540,inf,critical
160,2025-06-01 11:27:06.567188978,car_car,4.081237,0.289037,inf,critical
576,2025-06-01 20:10:31.812766075,car_car,3.984203,0.298700,inf,critical
205,2025-06-01 12:36:04.684072971,car_car,3.853472,0.344051,inf,critical
316,2025-06-01 14:57:48.365912914,car_car,9.703176,0.400082,inf,critical
